In [7]:
import pandas as pd

In [8]:
loan=pd.read_csv(r"../BankData//loan.csv", sep=";")
loan.head()

,loan_id,account_id,date,amount,duration,payments,status
0,5314,1787,930705,96396,12,8033.0,B
1,5316,1801,930711,165960,36,4610.0,A
2,6863,9188,930728,127080,60,2118.0,A
3,5325,1843,930803,105804,36,2939.0,A
4,7240,11013,930906,274740,60,4579.0,A


In [9]:
loan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   loan_id     682 non-null    int64  
 1   account_id  682 non-null    int64  
 2   date        682 non-null    int64  
 3   amount      682 non-null    int64  
 4   duration    682 non-null    int64  
 5   payments    682 non-null    float64
 6   status      682 non-null    object 
dtypes: float64(1), int64(5), object(1)
memory usage: 37.4+ KB


In [10]:
loan["account_id"].nunique()
loan[["account_id","loan_id"]].nunique()

account_id    682
loan_id       682
dtype: int64

In [11]:
loan["status"].value_counts()


status
C    403
A    203
D     45
B     31
Name: count, dtype: int64

In [12]:
loan["duration"].value_counts()

duration
60    145
24    138
48    138
12    131
36    130
Name: count, dtype: int64

In [13]:
loan.duplicated().sum()

0

In [14]:
loan["loan_id"] = loan["loan_id"].astype(str)
loan["account_id"] = loan["account_id"].astype(str)
loan["date"] = pd.to_datetime(loan["date"].astype(str).str.zfill(6), format="%y%m%d")
loan["amount"] = loan["amount"].astype(float)
loan["target"] = loan["status"].map({"A": 0, "B": 1, "C": 0, "D": 1})
loan["is_closed"] = loan["status"].isin(["A", "B"]).astype(int)

In [15]:
loan.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   loan_id     682 non-null    object        
 1   account_id  682 non-null    object        
 2   date        682 non-null    datetime64[ns]
 3   amount      682 non-null    float64       
 4   duration    682 non-null    int64         
 5   payments    682 non-null    float64       
 6   status      682 non-null    object        
 7   target      682 non-null    int64         
 8   is_closed   682 non-null    int32         
dtypes: datetime64[ns](1), float64(2), int32(1), int64(2), object(3)
memory usage: 45.4+ KB


In [16]:
loan.sample(5)

,loan_id,account_id,date,amount,duration,payments,status,target,is_closed
291,5415,2236,1996-09-26,139464.0,36,3874.0,D,1,0
69,6888,9265,1994-07-11,39576.0,12,3298.0,A,0,1
560,6861,9179,1998-02-28,215040.0,60,3584.0,C,0,0
57,6642,8027,1994-06-05,77640.0,12,6470.0,A,0,1
334,6878,9242,1997-01-24,23184.0,12,1932.0,A,0,1


## Data Cleaning & Target Labeling — `loan` Table

### 1. Data Cleaning & Transformation Log

| Column Name | Raw Type | Cleaned Type | Transformations Applied | Business / Risk Rationale |
| :--- | :---: | :---: | :--- | :--- |
| **`loan_id`** | `int64` | `str` | Cast to string object (`astype(str)`). | Treated as a categorical primary key; prevents accidental numerical math or inclusion in model correlation matrices. |
| **`account_id`** | `int64` | `str` | Cast to string object (`astype(str)`). | Relational foreign key connecting to `account.csv`. Verified 682 distinct accounts (strict 1:1 mapping). |
| **`date`** | `int64` | `datetime64[ns]` | Zero-padded to 6 digits (`str.zfill(6)`) and parsed with `%y%m%d`. | Enforces standard temporal boundary checks (`trans.date < loan.date`) to prevent future data leakage. |
| **`amount`** | `int64` | `float64` | Cast to continuous float (`astype(float)`). | Standardized financial currency metric to support division and ratio features (e.g., Debt-to-Income / PTI). |
| **`duration`** | `int64` | `int64` | Preserved as integer term. | Represents discrete contract terms in months (12, 24, 36, 48, 60). |
| **`payments`** | `float64` | `float64` | Preserved as continuous float. | Represents monthly debt obligation; essential for Capacity calculations. |
| **`status`** | `str` | `str` | Retained original categorical string (`A`, `B`, `C`, `D`). | Preserves original Czech bank contract codes for granular EDA and sub-sample filtering. |
| **`target`** | *New Feature* | `int64` | Mapped `{'A': 0, 'B': 1, 'C': 0, 'D': 1}`. | Ground truth label for Project A: `1` indicates default risk (`B`/`D`) and `0` indicates performing credit (`A`/`C`). |
| **`is_closed`** | *New Feature* | `int64` | Mapped `status.isin(['A', 'B'])` to binary flag (`0`/`1`). | Distinguishes closed historical contracts from active running loans. |

---

### 2. Validation & Integrity Audit Checklist

- [x] **Null Values:** 0 missing values detected across all 682 rows.
- [x] **Primary Key Uniqueness:** `loan_id` contains 682 unique values.
- [x] **Foreign Key Uniqueness:** `account_id` contains 682 unique values (confirms 1-to-1 account mapping).
- [x] **Duplicates:** `loan.duplicated().sum() == 0` (no duplicate rows).
- [x] **Categorical Values:** `duration` strictly contains `[12, 24, 36, 48, 60]` months.
- [x] **Target Distribution:**
  - **Good / Performing (`y = 0`):** 606 accounts (~88.85%)
  - **Default / Arrears (`y = 1`):** 76 accounts (~11.15%)

In [17]:
loan.to_csv("../cleanData/clean_loan.csv", sep=";", index=False)